## Phase 3 v2 — End-to-End Three-Head Classifier (Two-Stage Fine-Tuning)

This version revises the Phase 3 training strategy. Instead of jointly fine-tuning the
backbone, GAT, and heads from epoch 1 (as in the first version), training here is split
into two stages:

- **Stage 1** — backbone frozen, only the GAT and classification heads train, for 5 epochs
- **Stage 2** — backbone unfrozen, everything fine-tunes together, for up to 15 epochs
  with early stopping

The goal is to let the newly-initialized heads and the GAT stabilize on top of the
pretrained backbone before allowing gradients to flow into and disturb the backbone
itself — a standard warm-up strategy for transfer learning.

This cell installs dependencies, sets the device, and locates the Phase 1 and Phase 2
checkpoints. Both were found successfully.

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate
!pip install -q torch-geometric -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cu121.html

import os, glob, torch, numpy as np
from torch import nn
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

phase1_paths = glob.glob("/kaggle/input/**/mentalroberta_phase1_final", recursive=True)
PHASE1_PATH  = phase1_paths[0] if phase1_paths else None
PHASE2_PATH  = glob.glob("/kaggle/input/**/gat_phase2_best.pt", recursive=True)
PHASE2_PATH  = PHASE2_PATH[0] if PHASE2_PATH else None
print("Phase 1:", PHASE1_PATH)
print("Phase 2:", PHASE2_PATH)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.9 MB/s eta 0:00:00
Device: cuda
Phase 1: /kaggle/input/notebooks/shashwatkashyap12221/phase-1-mental-health-monitoring-system/mentalroberta_phase1_final
Phase 2: /kaggle/input/notebooks/shashwatkashyap12221/phase-2-mental-health-monitoring-system/gat_phase2_best.pt


## Step 1 — Load GoEmotions

Loads the same three raw GoEmotions CSV files used in the first version.
Combined shape: **211,225 rows × 37 columns** (one row per annotator rating, before
cleaning or deduplication).

In [ ]:
import pandas as pd

go_files = sorted(glob.glob("/kaggle/input/**/goemotions_*.csv", recursive=True))
print(go_files)
go_df = pd.concat([pd.read_csv(f) for f in go_files], ignore_index=True)
print(go_df.shape)

['/kaggle/input/datasets/shashwatkashyap12221/goemotions/goemotions_1.csv', '/kaggle/input/datasets/shashwatkashyap12221/goemotions/goemotions_2.csv', '/kaggle/input/datasets/shashwatkashyap12221/goemotions/goemotions_3.csv']
(211225, 37)


## Step 2 — Clean, Deduplicate, and Split

Unclear-labelled rows are dropped, then one row per unique comment is kept by taking the
max label value across all raters per emotion (same aggregation logic as the first
version). The result is split 85% / 7.5% / 7.5% into train / validation / test, using
`random_state=42` for reproducibility.

**Result:** 49,307 train / 4,351 validation / 4,351 test — identical split sizes to the
first version, confirming the data pipeline hasn't changed, only the training strategy.

In [ ]:
from sklearn.model_selection import train_test_split

EMOTION_COLS = ['admiration','amusement','anger','annoyance','approval','caring',
                'confusion','curiosity','desire','disappointment','disapproval',
                'disgust','embarrassment','excitement','fear','gratitude','grief',
                'joy','love','nervousness','optimism','pride','realization',
                'relief','remorse','sadness','surprise','neutral']

go_clean = go_df[go_df["example_very_unclear"] == False].copy()
agg = go_clean.groupby("id").agg(
    {**{"text":"first"}, **{c:"max" for c in EMOTION_COLS}}
).reset_index()

train_df, temp_df = train_test_split(agg, test_size=0.15, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
print(train_df.shape, val_df.shape, test_df.shape)

(49307, 30) (4351, 30) (4351, 30)


## Step 3 — Tokenizer and DataLoaders

Reuses the Phase 1 MentalRoBERTa tokenizer, tokenizing each comment to a fixed 128
tokens (padding + truncation). Batch sizes: 16 for training, 32 for validation/test.
This matches the first version exactly.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PHASE1_PATH)

class GoEmotionsDataset(Dataset):
    def __init__(self, df, max_len=128):
        self.texts  = df["text"].tolist()
        self.labels = df[EMOTION_COLS].values.astype(np.float32)
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True,
                        padding="max_length", max_length=self.max_len,
                        return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx])
        }

train_ds = GoEmotionsDataset(train_df)
val_ds   = GoEmotionsDataset(val_df)
test_ds  = GoEmotionsDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
test_loader  = DataLoader(test_ds,  batch_size=32)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Train: 49307 | Val: 4351 | Test: 4351


## Step 4 — Phase3v2Model Architecture

Architecturally identical to the first version:

- **Backbone**: MentalRoBERTa (Phase 1 checkpoint) → 768-dim token embeddings
- **GAT**: `gat1` (768 → 4 heads × 256, concatenated) → `gat2` (1024 → 2 heads → 256,
  averaged), using sequential token-to-token edges cached per (batch, seq_len)
- **Emotion head**: 256 → 128 → 28 (multi-label)
- **MH head**: 256 → 64 → 2 (binary)
- **Severity head**: 256 → 64 → 1, frozen (still pending DAIC-WOZ)

The UNEXPECTED/MISSING key warnings on load are expected — Phase 1 saved a sequence
classifier head that this base model doesn't use, and a fresh pooler is initialized
in its place. No architectural changes from v1; the difference in this version is
entirely in **how** training is scheduled (see Step 6 below).

In [ ]:
from torch_geometric.nn import GATConv

class Phase3v2Model(nn.Module):
    def __init__(self, backbone_path, num_emotions=28):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone_path)
        self.gat1 = GATConv(768,  256, heads=4, concat=True)
        self.gat2 = GATConv(1024, 256, heads=2, concat=False)

        self.emotion_head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, num_emotions)
        )
        self.mh_head = nn.Sequential(
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 2)
        )
        self.severity_head = nn.Sequential(
            nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid()
        )
        for p in self.severity_head.parameters():
            p.requires_grad = False

        self._edge_cache = {}

    def _get_edges(self, B, S):
        key = (B, S)
        if key not in self._edge_cache:
            src, dst = [], []
            for b in range(B):
                offset = b * S
                src.append(torch.arange(offset, offset + S - 1))
                dst.append(torch.arange(offset + 1, offset + S))
            self._edge_cache[key] = torch.stack(
                [torch.cat(src), torch.cat(dst)]
            )
        return self._edge_cache[key].to(device)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids,
                            attention_mask=attention_mask)
        B, S, H = out.last_hidden_state.shape
        x = out.last_hidden_state.reshape(B * S, H)
        edge_index = self._get_edges(B, S)
        x = torch.relu(self.gat1(x, edge_index))
        x = torch.relu(self.gat2(x, edge_index))
        cls_idx = torch.arange(B, device=x.device) * S
        vec = x[cls_idx]
        return self.emotion_head(vec), self.mh_head(vec), \
               self.severity_head(vec).squeeze(-1)

model = Phase3v2Model(PHASE1_PATH, num_emotions=len(EMOTION_COLS)).to(device)
print("Model ready")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: /kaggle/input/notebooks/shashwatkashyap12221/phase-1-mental-health-monitoring-system/mentalroberta_phase1_final
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.bias      | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model ready


## Step 5 — Transfer Phase 2 GAT Weights

Same goal as the first version — load the Phase 2 GAT's learned weights — but the
matching logic is more explicit here: an exact `key_map` dictionary lists all 8 expected
GAT parameter names (att_src, att_dst, bias, lin.weight for each of gat1/gat2) instead of
relying on automatic shape-matching across the whole checkpoint.

**Result: 8/8 layers matched.** Functionally the same outcome as the first version's
"8/10" (the other 2 keys there were the old Phase 2 classifier head, which was never
meant to transfer), but this version's explicit key list makes that intent clear in the
code itself rather than needing a comment to explain it.

In [ ]:
if PHASE2_PATH:
    gat_ckpt   = torch.load(PHASE2_PATH, map_location=device)
    model_dict = model.state_dict()
    key_map = {
        "gat1.att_src":"gat1.att_src", "gat1.att_dst":"gat1.att_dst",
        "gat1.bias":"gat1.bias",       "gat1.lin.weight":"gat1.lin.weight",
        "gat2.att_src":"gat2.att_src", "gat2.att_dst":"gat2.att_dst",
        "gat2.bias":"gat2.bias",       "gat2.lin.weight":"gat2.lin.weight",
    }
    matched = 0
    for p2k, mk in key_map.items():
        if p2k in gat_ckpt and mk in model_dict:
            if gat_ckpt[p2k].shape == model_dict[mk].shape:
                model_dict[mk] = gat_ckpt[p2k]
                matched += 1
    model.load_state_dict(model_dict)
    print(f"Phase 2 GAT weights loaded: {matched}/8")

Phase 2 GAT weights loaded: 8/8


## Step 6 — Loss Functions and Evaluation Helper

Same two losses as the first version (`BCEWithLogitsLoss` for emotions,
`CrossEntropyLoss` for MH category, combined as 0.6/0.4). A `compute_metrics()` helper
is added here so the same validation-F1 calculation (0.3 sigmoid threshold, macro F1)
can be reused identically across both training stages below, instead of being
duplicated inline as in the first version.

In [ ]:
bce = nn.BCEWithLogitsLoss()
ce  = nn.CrossEntropyLoss()

def compute_metrics(loader):
    model.eval()
    all_preds, all_labs = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            emo_logits, _, _ = model(ids, mask)
            probs = torch.sigmoid(emo_logits).cpu().numpy()
            all_preds.append((probs >= 0.3).astype(int))
            all_labs.append(batch["labels"].numpy())
    return f1_score(np.vstack(all_labs), np.vstack(all_preds),
                    average="macro", zero_division=0)

print("Loss functions ready")

Loss functions ready


## Step 7 — Stage 1: Backbone Frozen (5 Epochs)

The backbone is frozen (`requires_grad = False`), so only the GAT and the emotion/MH
heads are trained, using a much higher learning rate (5e-4) since these layers are
starting from scratch (GAT is warm-started from Phase 2, heads are random). The best
checkpoint by validation macro-F1 is saved after each improving epoch.

**Results:**

| Epoch | Train Loss | Val F1 | Best? |
|---|---|---|---|
| 1 | 0.1192 | 0.4902 | ✓ |
| 2 | 0.1138 | 0.4882 | |
| 3 | 0.1126 | 0.4897 | |
| 4 | 0.1118 | 0.5044 | ✓ |
| 5 | 0.1112 | 0.5071 | ✓ (best) |

Stage 1 alone — with the backbone still frozen — already reaches **0.5071 val F1**,
slightly higher than the first version's entire 3-epoch joint-training run (0.5008).
This suggests the GAT and heads benefit from a few epochs of stable, focused training
before the backbone starts moving underneath them.

In [ ]:
# ── STAGE 1: backbone frozen ──────────────────────────────────────────────
print("=" * 60)
print("STAGE 1 — Backbone FROZEN, training heads only (5 epochs)")
print("=" * 60)

for param in model.backbone.parameters():
    param.requires_grad = False

optimizer_s1 = torch.optim.AdamW([
    {"params": model.gat1.parameters(),          "lr": 5e-4},
    {"params": model.gat2.parameters(),          "lr": 5e-4},
    {"params": model.emotion_head.parameters(),  "lr": 5e-4},
    {"params": model.mh_head.parameters(),       "lr": 5e-4},
], weight_decay=0.01)

S1_EPOCHS = 5
best_f1   = 0

for epoch in range(S1_EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labs = batch["labels"].to(device)

        optimizer_s1.zero_grad()
        emo_logits, mh_logits, _ = model(ids, mask)
        mh_labs = (labs.sum(dim=1) > 0).long()
        loss = 0.6 * bce(emo_logits, labs) + 0.4 * ce(mh_logits, mh_labs)
        loss.backward()
        optimizer_s1.step()
        total_loss += loss.item()

    f1 = compute_metrics(val_loader)
    print(f"[S1] Epoch {epoch+1}/{S1_EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Val F1: {f1:.4f}", end="")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "/kaggle/working/phase3v2_s1_best.pt")
        print(" ✓")
    else:
        print()

print(f"\nStage 1 complete. Best F1: {best_f1:.4f}")

STAGE 1 — Backbone FROZEN, training heads only (5 epochs)
[S1] Epoch 1/5 | Loss: 0.1192 | Val F1: 0.4902 ✓
[S1] Epoch 2/5 | Loss: 0.1138 | Val F1: 0.4882
[S1] Epoch 3/5 | Loss: 0.1126 | Val F1: 0.4897
[S1] Epoch 4/5 | Loss: 0.1118 | Val F1: 0.5044 ✓
[S1] Epoch 5/5 | Loss: 0.1112 | Val F1: 0.5071 ✓

Stage 1 complete. Best F1: 0.5071


## Step 8 — Stage 2: Full Fine-Tuning (Up to 15 Epochs, Early Stopping)

The Stage 1 best checkpoint is reloaded, the backbone is unfrozen, and training
continues with differential learning rates (backbone 2e-5, GAT/heads 5e-5 — same rates
as the first version's single-stage run) for up to 15 epochs, with early stopping after
5 epochs of no improvement.

**Results:**

| Epoch | Train Loss | Val F1 | Note |
|---|---|---|---|
| 1 | 0.1132 | 0.5033 | best |
| 2 | 0.1071 | 0.5118 | best |
| 3 | 0.1008 | 0.5055 | no improve 1/5 |
| 4 | 0.0946 | 0.5121 | best |
| 5 | 0.0883 | 0.5002 | no improve 1/5 |
| 6 | 0.0819 | 0.5007 | no improve 2/5 |
| 7 | 0.0757 | 0.4999 | no improve 3/5 |
| 8 | 0.0698 | 0.4967 | no improve 4/5 |
| 9 | 0.0641 | 0.4920 | no improve 5/5 → stopped |

Training loss keeps falling every epoch, but validation F1 peaks at epoch 4 (0.5121)
and then steadily declines — a textbook overfitting curve once the backbone is
unfrozen. Early stopping correctly halts training at epoch 9 and the epoch-4 checkpoint
is kept. **Overall improvement across both stages: 0.5071 → 0.5121.**

In [ ]:
# ── STAGE 2: unfreeze backbone ────────────────────────────────────────────
print("=" * 60)
print("STAGE 2 — Backbone UNFROZEN, full fine-tuning (15 epochs)")
print("=" * 60)

# load best stage 1 checkpoint before unfreezing
model.load_state_dict(torch.load("/kaggle/working/phase3v2_s1_best.pt"))

for param in model.backbone.parameters():
    param.requires_grad = True

optimizer_s2 = torch.optim.AdamW([
    {"params": model.backbone.parameters(),      "lr": 2e-5},  # much lower
    {"params": model.gat1.parameters(),          "lr": 5e-5},
    {"params": model.gat2.parameters(),          "lr": 5e-5},
    {"params": model.emotion_head.parameters(),  "lr": 5e-5},
    {"params": model.mh_head.parameters(),       "lr": 5e-5},
], weight_decay=0.01)

S2_EPOCHS  = 15
patience   = 5
no_improve = 0
best_f1_s2 = 0

for epoch in range(S2_EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labs = batch["labels"].to(device)

        optimizer_s2.zero_grad()
        emo_logits, mh_logits, _ = model(ids, mask)
        mh_labs = (labs.sum(dim=1) > 0).long()
        loss = 0.6 * bce(emo_logits, labs) + 0.4 * ce(mh_logits, mh_labs)
        loss.backward()
        optimizer_s2.step()
        total_loss += loss.item()

    f1 = compute_metrics(val_loader)
    print(f"[S2] Epoch {epoch+1}/{S2_EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Val F1: {f1:.4f}", end="")

    if f1 > best_f1_s2:
        best_f1_s2 = f1
        no_improve = 0
        torch.save(model.state_dict(), "/kaggle/working/phase3v2_best.pt")
        print(" ✓ best")
    else:
        no_improve += 1
        print(f" (no improve {no_improve}/{patience})")
        if no_improve >= patience:
            print(f"\nEarly stopping at S2 epoch {epoch+1}")
            break

print(f"\nStage 2 complete. Best F1: {best_f1_s2:.4f}")
print(f"Overall improvement: {best_f1:.4f} → {best_f1_s2:.4f}")

STAGE 2 — Backbone UNFROZEN, full fine-tuning (15 epochs)
[S2] Epoch 1/15 | Loss: 0.1132 | Val F1: 0.5033 ✓ best
[S2] Epoch 2/15 | Loss: 0.1071 | Val F1: 0.5118 ✓ best
[S2] Epoch 3/15 | Loss: 0.1008 | Val F1: 0.5055 (no improve 1/5)
[S2] Epoch 4/15 | Loss: 0.0946 | Val F1: 0.5121 ✓ best
[S2] Epoch 5/15 | Loss: 0.0883 | Val F1: 0.5002 (no improve 1/5)
[S2] Epoch 6/15 | Loss: 0.0819 | Val F1: 0.5007 (no improve 2/5)
[S2] Epoch 7/15 | Loss: 0.0757 | Val F1: 0.4999 (no improve 3/5)
[S2] Epoch 8/15 | Loss: 0.0698 | Val F1: 0.4967 (no improve 4/5)
[S2] Epoch 9/15 | Loss: 0.0641 | Val F1: 0.4920 (no improve 5/5)

Early stopping at S2 epoch 9

Stage 2 complete. Best F1: 0.5121
Overall improvement: 0.5071 → 0.5121


## Step 9 — Final Test Evaluation

The best Stage 2 checkpoint (val F1 = 0.5121) is evaluated on the held-out 4,351-sample
test set.

**Overall:** Macro F1 **0.51**, Micro F1 **0.59**, Samples F1 **0.61**.

Compared to the first version's test results (Macro F1 0.49 / Micro 0.59 / Samples
0.61), micro and samples F1 are unchanged, while macro F1 improved by 2 points — driven
mostly by better recall on rare classes (e.g. pride F1 0.02 → 0.21, grief 0.08 → 0.13,
relief 0.16 → 0.28), at a small precision cost on some mid-frequency classes.

**"Professor's trick"** — for the first 5 validation examples, instead of showing all 28
sigmoid probabilities, only the single highest-scoring emotion and its confidence are
printed. This is purely a presentation/inference convenience for demoing the model
one-sample-at-a-time — it doesn't change training, the saved metrics, or the checkpoint
in any way.

Final checkpoints (`phase3v2_final.pt`, `phase3v2_tokenizer/`) are saved for downstream
use.

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/phase3v2_best.pt"))
model.eval()

all_preds, all_labs = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        emo_logits, _, _ = model(ids, mask)
        probs = torch.sigmoid(emo_logits).cpu().numpy()
        all_preds.append((probs >= 0.3).astype(int))
        all_labs.append(batch["labels"].numpy())

print("=== PHASE 3 v2 — TEST RESULTS ===")
report = classification_report(
    np.vstack(all_labs), np.vstack(all_preds),
    target_names=EMOTION_COLS, zero_division=0, output_dict=True
)
print(classification_report(
    np.vstack(all_labs), np.vstack(all_preds),
    target_names=EMOTION_COLS, zero_division=0
))

# professor's trick: for inference, show only the top-scoring emotion
print("\n=== PROFESSOR'S TRICK — Top emotion per sample (first 5 val examples) ===")
model.eval()
sample_batch = next(iter(val_loader))
with torch.no_grad():
    emo_logits, _, _ = model(sample_batch["input_ids"].to(device),
                              sample_batch["attention_mask"].to(device))
probs = torch.sigmoid(emo_logits).cpu().numpy()
for i in range(5):
    top_idx   = probs[i].argmax()
    top_score = probs[i][top_idx]
    print(f"  Sample {i+1}: '{EMOTION_COLS[top_idx]}' (confidence: {top_score:.3f})")

torch.save(model.state_dict(), "/kaggle/working/phase3v2_final.pt")
tokenizer.save_pretrained("/kaggle/working/phase3v2_tokenizer")
print("\nPhase 3 v2 complete. All checkpoints saved.")

Testing: 100%|██████████| 136/136 [00:28<00:00,  4.70it/s]


=== PHASE 3 v2 — TEST RESULTS ===
                precision    recall  f1-score   support

    admiration       0.67      0.71      0.69       764
     amusement       0.82      0.71      0.76       408
         anger       0.49      0.60      0.54       417
     annoyance       0.44      0.59      0.50       712
      approval       0.50      0.52      0.51       990
        caring       0.49      0.47      0.48       343
     confusion       0.54      0.59      0.57       396
     curiosity       0.63      0.77      0.69       432
        desire       0.44      0.46      0.45       181
disappointment       0.40      0.46      0.43       490
   disapproval       0.43      0.68      0.53       615
       disgust       0.42      0.53      0.47       303
 embarrassment       0.44      0.27      0.34       132
    excitement       0.36      0.46      0.40       318
          fear       0.54      0.58      0.56       167
     gratitude       0.74      0.74      0.74       385
         grie